In [1]:
from random import shuffle
from time import sleep
from PIL import Image
import requests
import json
import csv
import os

metObjects_folder='MetObjects'

# first let's load something from unsplash
file='MetObjects.csv'
filepath=os.path.join(metObjects_folder,file)

# load the file as a list of dicts
list_artwork=[]
with open(filepath, 'r') as c:
    r = csv.DictReader(c)
    list_artwork =list(r)

# take a look at the data
print(f"Loaded {len(list_artwork)} artworks, details of first artwork:\n")
for key, value in list_artwork[0].items():
    print(f"{key}: {value}")

Loaded 484956 artworks, details of first artwork:

﻿Object Number: 1979.486.1
Is Highlight: False
Is Timeline Work: False
Is Public Domain: False
Object ID: 1
Gallery Number: 
Department: The American Wing
AccessionYear: 1979
Object Name: Coin
Title: One-dollar Liberty Head Coin
Culture: 
Period: 
Dynasty: 
Reign: 
Portfolio: 
Constituent ID: 16429
Artist Role: Maker
Artist Prefix:  
Artist Display Name: James Barton Longacre
Artist Display Bio: American, Delaware County, Pennsylvania 1794–1869 Philadelphia, Pennsylvania
Artist Suffix:  
Artist Alpha Sort: Longacre, James Barton
Artist Nationality: American
Artist Begin Date: 1794      
Artist End Date: 1869      
Artist Gender: 
Artist ULAN URL: http://vocab.getty.edu/page/ulan/500011409
Artist Wikidata URL: https://www.wikidata.org/wiki/Q3806459
Object Date: 1853
Object Begin Date: 1853
Object End Date: 1853
Medium: Gold
Dimensions: Dimensions unavailable
Credit Line: Gift of Heinz L. Stoppelmann, 1979
Geography Type: 
City: 
State: 

In [2]:
# there is a lot of data in here that we don't want so let's start by removing any that aren't public domain
public_domain = [a for a in list_artwork if a["Is Public Domain"]=='True']

# take a look at the data again
print(f"Loaded {len(public_domain)} artworks, details of first artwork:\n")
for key, value in public_domain[0].items():
    print(f"{key}: {value}")

Loaded 248472 artworks, details of first artwork:

﻿Object Number: 1970.289.6
Is Highlight: False
Is Timeline Work: False
Is Public Domain: True
Object ID: 34
Gallery Number: 774
Department: The American Wing
AccessionYear: 1970
Object Name: Clock
Title: Acorn Clock
Culture: American
Period: 
Dynasty: 
Reign: 
Portfolio: 
Constituent ID: 108
Artist Role: Maker
Artist Prefix:  
Artist Display Name: Forestville Manufacturing Company
Artist Display Bio: 1835–1853
Artist Suffix:  
Artist Alpha Sort: Forestville Manufacturing Company 
Artist Nationality: American
Artist Begin Date: 1835      
Artist End Date: 1853      
Artist Gender: 
Artist ULAN URL: 
Artist Wikidata URL: 
Object Date: 1847–50
Object Begin Date: 1847
Object End Date: 1850
Medium: Mahogany, laminated
Dimensions: 24 3/8 x 14 5/8 x 5 1/8 in. (61.9 x 37.1 x 13 cm)
Credit Line: Gift of Mrs. Paul Moore, 1970
Geography Type: Made in
City: Bristol
State: 
County: 
Country: United States
Region: 
Subregion: 
Locale: 
Locus: 
Excav

In [3]:
# we're down to 250k, we could take randomly from this but ideally we want to minimise high quality photographs as unsplash serves that purposes
# so lets have a look at the Medium tags, we can filter out any without a Medium
mediums = []
[mediums.append(v["Medium"].strip().lower()) for v in public_domain if v["Medium"].strip() and v["Medium"].strip().lower() not in mediums]

# how many mediums do we have?
print(f"We have {len(mediums)} different mediums.\n")

We have 37448 different mediums.



In [4]:
# Let's have a look at some of them and start building a list of things we want to exclude
# create a new variable first so we don't have to rerun the above if we go a bit too far
filtered_mediums=mediums

# we can keep running this and adding to the exclusions until we're down to a reasonable number
exclude_mediums = [
    'none','glass','basalt','iron','lusterware','earthenware','bronze','maple','mahogany','brass','rattan',
    'walnut','pine','porcelain','plast','wood','stone','poplar','copper','ivory','diamond','emerald',
    'terracotta','silver','beech','pewter','oak','gold','mache','metal','cherry','amber','tin','ash',
    'tortoiseshell','cedar','faience','ceramic','flint','birch','leaf','jasperware','sardonyx','wax',
    'steel','panel','wax','bone','lead','butternut','alabaster','cotton','wool','linen','silk','woven',
    'vulcanite','engraving','embroidered','trimming','leather','horn','pottery','slate','schist',
    'steatite','granulite','granite','stucco','carved','marble','gypsum','andesite','phyllite','argillite',
    'agate','clay','malachite','bowenite','aquamarine','nephrite','twill','crystal','turquoise',
    'carnelian','jasper','lazuli','bamboo','pearl','jadeite','quartz','buncheong','sgraffito','jadeitite',
    'fluorite','cypress','dolomite','hemp','papier-mâché','amethyst','gourd','calcified','fluorspar',
    'fibrolite','carborundum','corundum','garnet','no medium','macramé','hair','lace','obsidian','rhyolite',
    'basketry','porcelan','topaz','feldspar','granitodiorite','oyster shell','porphyry','antler','stamp',
    'seal','medal','button','pitcher'
]
filtered_mediums = [m for m in mediums if not any([e in m.lower() for e in exclude_mediums])]

print(f"We have {len(filtered_mediums)} different filtered mediums.\n")
# we will shuffle and show the first 50 so we can keep running and adding to the above until we're happy
shuffle(filtered_mediums)
for m in filtered_mediums[:50]:
    print(m)

We have 6654 different filtered mediums.

purple, pink, black, and white chalk
pen and brown ink, over graphite underdrawing and compass and ruled construction lines in pen and brown ink and graphite
commerical lithograph with embossment on folded sheet
pen and ink, watercolor on wove paper
three-color assembly print
red chalk over graphite underdrawing
gouache. framing line in brush and black ink.
pen and brown ink, reworked with black ink (recto); pen and brown and black ink (verso)
tempera and gesso on canvas
pen and light brown ink, selectively reinforced with dark brown ink, over black chalk underdrawing. border lines in the same brown ink (recto). fragment of architectural design with engaged corinthian columns. pen and light brown ink over ruling in black chalk (verso). on off-white paper
pen and brown ink, watercolor, and gouache
red hematite
etching, printed in black ink on heavy pale gray textured paper with (faded) blue fibers; third state of four (glasgow)
etching; third ed

In [5]:
# This has brought the list down somewhat, little of which seems to be photography of physical media, so lets use it to shorten our list of public domain works
public_domain_filtered = [a for a in public_domain if not any([e in a["Medium"].strip().lower() for e in exclude_mediums])]

# take a look at the data again
# this time shuffle first, look at a few more and include the resource link this time
print(f"Loaded {len(public_domain_filtered)} artworks, details of first 10 works:\n")
shuffle(public_domain_filtered)
for item in public_domain_filtered[:10]:
    print(f"Medium: {item['Medium']}")
    print(f"Link Resource: {item['Link Resource']}\n")

Loaded 55305 artworks, details of first 10 works:

Medium: Watercolor, gouache, black ink, and graphite on white wove paper
Link Resource: http://www.metmuseum.org/art/collection/search/13176

Medium: Commercial color lithograph
Link Resource: http://www.metmuseum.org/art/collection/search/631407

Medium: 
Link Resource: http://www.metmuseum.org/art/collection/search/215297

Medium: Etching; first state of three (Bartsch)
Link Resource: http://www.metmuseum.org/art/collection/search/401359

Medium: Etching
Link Resource: http://www.metmuseum.org/art/collection/search/408626

Medium: Commercial color lithograph
Link Resource: http://www.metmuseum.org/art/collection/search/712318

Medium: Commercial color lithograph
Link Resource: http://www.metmuseum.org/art/collection/search/640201

Medium: Pen and gray ink, watercolor
Link Resource: http://www.metmuseum.org/art/collection/search/397998

Medium: Black chalk
Link Resource: http://www.metmuseum.org/art/collection/search/338328

Medium: A

In [9]:
# most of these look like they have the kind of distortions our models were struggling with so now let's download some of them
download_folder='images/'
# set how many images to download
num_to_download=5000
# maximum failures
max_failures=25
# and a few variables to keep track of successes and failures
loaded=0
skipped=0
failed=0

# the primaryImage URI isn't included in the CSV so we'll need to use the Met's API here
# see https://metmuseum.github.io/#search

# set the endpoint, we can add the object ID later
endpoint = "https://collectionapi.metmuseum.org/public/collection/v1/objects/"

def download_artwork(artwork,objectID):
    # we need the URL and the name from the JSON
    photo_url=artwork["primaryImage"]
    name=artwork["objectName"].lower().strip().replace(' ','_')
    # build the name and filepath from the name and ID of the object
    filename=f"{name}_{objectID}.jpg"
    filepath=os.path.join(download_folder,filename)
    # download the file (if it doesn't already exist)
    if os.path.isfile(filepath):
        return True
    else:
        try:
            # the met have some very small images not suitable for our work, so using PIL this time so we can check dimensions
            # see https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Image.save
            downloaded_image = Image.open(requests.get(photo_url, stream=True).raw)
            # we will skip over smaller images
            if min(downloaded_image.size)>512:
                # note that format is derived from filename
                downloaded_image.save(filepath)
            else:
                return None
            return True
        except Exception as e:
            print(f"Download failed for {photo_url} with the following exception: {e}")
            return False

# we'll create a copy of the list this time and pop from it each loop until we're done or it's exhausted
images_to_download = public_domain_filtered.copy()
while loaded<num_to_download and failed<=max_failures and images_to_download:
    item = images_to_download.pop()
    objectID = item["Object ID"]
    response = requests.get(endpoint+objectID)
    if response.ok:
        artwork = json.loads(response.content)
        # we only want to download if it's public domain, so adding a check in case we make a mistake
        if artwork["isPublicDomain"]:
            success = download_artwork(artwork,objectID)
            if success:
                loaded+=1
            elif success is None:
                skipped+=1
            else:
                failed+=1
        else:
            print(f"Download aborted for Object ID {objectID} as public domain status {artwork['isPublicDomain']}")
            skipped+=1
        # adding a sleep here as I don't want to hit whatever request limit the met have set
        sleep(0.25)
    else:
        print(f"Failed on object ID {objectID} with status code {response.status_code}: {response.reason}")

print(f"\n{loaded} images downloaded and saved in {os.path.join(download_folder)}, {failed} downloads failed and {skipped} were skipped.")

Failed on object ID 43002 with status code 404: Not Found
Failed on object ID 61959 with status code 404: Not Found
Download failed for https://images.metmuseum.org/CRDImages/dp/original/DR506.jpg with the following exception: cannot identify image file <_io.BytesIO object at 0x7f0d48189710>
Failed on object ID 42788 with status code 404: Not Found
Failed on object ID 43164 with status code 404: Not Found
Failed on object ID 42801 with status code 404: Not Found
Download aborted for Object ID 830281 as public domain status False
Failed on object ID 43168 with status code 404: Not Found
Failed on object ID 42877 with status code 404: Not Found
Failed on object ID 42825 with status code 404: Not Found
Download failed for https://images.metmuseum.org/CRDImages/dp/original/DP886892.jpg with the following exception: cannot identify image file <_io.BytesIO object at 0x7f0d48188a90>
Failed on object ID 42969 with status code 404: Not Found
Download failed for  with the following exception: In